## 🚀 UN Space Object Index Data Extraction + EDA 

Automates the extraction, cleaning, and exploratory analysis of launch and mission data from a NASA infinite-scroll web resource to inform space-tech market intelligence and infrastructure strategy.


### TL;DR Executive Summary
**3 Findings**:  
1. Successfully built an automated scraper for UNs mission listings using infinite-scroll handling.  
2. Cleaned and structured the dataset into consistent formats for mission names, launch dates, locations, and mission objectives.  
3. Conducted exploratory data analysis revealing patterns in mission frequency, geographic launch distribution, and thematic mission types.

**2 Implications**:  
1. Enables ongoing, low-effort tracking of the UNs tracked object list.  
2. Establishes a reusable pipeline for other space agencies’ open portals.

**1 Recommendation**:  
Extend this workflow to include launch success metrics, payload mass, and satellite type for richer strategic correlation.



### 🎯 Problem Statement & Decision Context

- ***Scope***: NASA missions page, infinite scroll, structured tabular extraction, CSV/GeoJSON storage, EDA in Python.  
- **Out of Scope**: Real-time mission tracking, orbital mechanics calculations, and EO raster integration (covered in later projects).  
- **Success Criteria**: Fully automated data extraction script + cleaned dataset + EDA visualizations revealing at least three actionable patterns.



### 👥 Use Cases
- **Primary Stakeholders**:  
  - Aerospace startups (Skyroot, Pixxel) for competitive benchmarking and comparitive analysis. 
  - Space policy think tanks for mission diversity analysis
  - Infrastructure planners for launch site capacity planning



### 🗂 Data Card
- **Source**: NASA Launch/Mission website (infinite scroll endpoint)  
- **Method**: Python requests with dynamic content loading handling  
- **License**: Public domain (US Government works)  
- **Update Frequency**: Daily/Weekly (can be scheduled)  
- **Key Attributes**:  
  - `mission_name` (string)  
  - `launch_date` (datetime)  
  - `launch_location` (string)  
  - `mission_type` (categorical)  
  - `mission_summary` (text)

- **Known Limitations**:  
  - Data may omit classified missions  
  - Inconsistent mission type labels require standardization

---

### 🔍 Method Overview
1. ***Data Extraction***  
   - Automated infinite-scroll loading until all records loaded  
   - HTML parsing & structured field extraction  
2. ***Data Cleaning***  
   - Standardizing dates, normalizing location names, deduplicating records  
3. ***Exploratory Data Analysis***  
   - Launch frequency by year/quarter  
   - Launch sites distribution mapping  
   - Mission type breakdown  
4. ***Output Preparation***  
   - CSV for tabular use  
   - GeoJSON for GIS integration


In [ ]:
# Inputs: none. Process: import libs and set BASE URL. Output: env ready for UNOOSA JSON requests.
import json, time, urllib.parse, requests
import pandas as pd
from tqdm import tqdm

BASE = "https://www.unoosa.org/oosa/osoindex/waxs-search.json"

In [ ]:
# Inputs: filters/sort/startAt. Process: build search payload. Output: criteria dict for UNOOSA API.
def build_criteria(filters=None, start_at=0, sortings=None):
    return {
        "filters": filters or [],
        "sortings": sortings or [{"fieldName":"object.launch.dateOfLaunch_s1","dir":"desc"}],
        "startAt": int(start_at),
    }

In [ ]:
# Inputs: session/criteria. Process: call UNOOSA JSON endpoint. Output: total found and page results.
def fetch_page(session: requests.Session, criteria: dict, cookies=None, timeout=30):
    crit = json.dumps(criteria, separators=(",", ":"))
    url = f"{BASE}?criteria={urllib.parse.quote(crit, safe='')}"
    headers = {
        "Accept": "application/json, text/plain, */*",
        "Referer": "https://www.unoosa.org/oosa/osoindex/search-ng.jspx?lf_id=",
        "User-Agent": "Mozilla/5.0",
    }
    r = session.get(url, headers=headers, cookies=cookies or {}, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    found = data.get("found", data.get("responseData", {}).get("found"))
    results = data.get("results", data.get("responseData", {}).get("results", []))
    return int(found or 0), results or []

In [ ]:
# Inputs: nested dict. Process: flatten nested keys. Output: 1-level dict for DataFrame rows.
def flatten(d, parent="", sep="."):
    out = {}
    for k, v in (d or {}).items():
        nk = f"{parent}{sep}{k}" if parent else k
        if isinstance(v, dict):
            out.update(flatten(v, nk, sep))
        elif isinstance(v, list):
            out[nk] = json.dumps(v, ensure_ascii=False)
        else:
            out[nk] = v
    return out

In [ ]:
# Inputs: filters/sort/limit. Process: page UNOOSA with progress. Output: full DataFrame of results.
def fetch_all(filters=None, sortings=None, cookies=None, sleep_s=0.5, limit=None):
    session = requests.Session()
    found, results = fetch_page(session, build_criteria(filters, 0, sortings), cookies=cookies)
    page_size = len(results)
    rows = [flatten(r) for r in results]

    # cap progress to limit if provided
    target = min(found, limit) if limit is not None else found
    from tqdm import tqdm
    pbar = tqdm(total=target, desc="Download UNOOSA", unit="rows")
    pbar.update(min(len(results), target))

    if limit is not None and len(rows) >= limit:
        pbar.close()
        return pd.DataFrame(rows[:limit])

    start_at = page_size
    while start_at < found and page_size > 0:
        time.sleep(sleep_s)
        found2, results = fetch_page(session, build_criteria(filters, start_at, sortings), cookies=cookies)
        found = max(found, found2 or found)
        if not results:
            break
        rows.extend(flatten(r) for r in results)
        page_size = len(results)
        start_at += page_size

        # update and stop when limit reached
        if limit is not None:
            remaining = max(0, limit - (pbar.n))
            pbar.update(min(len(results), remaining))
            if len(rows) >= limit:
                pbar.close()
                return pd.DataFrame(rows[:limit])
        else:
            pbar.update(len(results))

    pbar.close()

In [ ]:
# Inputs: optional cookies. Process: fetch all UNOOSA rows. Output: df_unoosa preview.
# Optional: cookies = {"JSESSIONID":"...", "_ga":"...", "UNOOSA-NSLB":"..."}  # redact secrets
cookies = None

df_unoosa = fetch_all(
    filters=[], 
    sortings=[{"fieldName":"object.launch.dateOfLaunch_s1","dir":"desc"}],
    cookies=cookies,
)
df_unoosa.head()

Download UNOOSA: 100%|██████████| 21289/21289 [24:28<00:00, 14.49rows/s]


,id,uri,values.object.internationalDesignator_s1,values.object.internationalDesignator@official_s1,values.object.nationalDesignator_s1,values.object.nameOfSpaceObjectIno_s1,values.object.nameOfSpaceObjectO_s1,values.object.launch.stateOfRegistry_s1,values.object.launch.stateOfRegistry@official_s1,values.object.launch.dateOfLaunch_s1,...,values.object.launch.dateOfLaunch@official_s1,values.object.status.dateOfDecay@official_s1,values.object.functionOfSpaceObject_s1,values.object.remark_s1,values.object.status.webSite_s1,values.object.unRegistration.registrationDocuments.document@uri_s,values.object.unRegistration.registrationDocuments.document..document.symbol_s,values.object.status.gsoLocation@official_s1,values.object.unRegistration.decayDocuments.document@uri_s,values.object.unRegistration.decayDocuments.document..document.symbol_s
0,"102,en,/osoindex/data/objects/2025/2025-085q_2...",/osoindex/data/objects/2025/2025-085q_24495.html,2025-085Q,false,,STARLINK 33861,,USA,false,2025-04-28,...,false,false,------,Not registered with the United Nations. Date o...,,NaN,NaN,NaN,NaN,NaN
1,"102,en,/osoindex/data/objects/2025/2025-085s_2...",/osoindex/data/objects/2025/2025-085s_24497.html,2025-085S,false,,STARLINK 33887,,USA,false,2025-04-28,...,false,false,------,Not registered with the United Nations. Date o...,,NaN,NaN,NaN,NaN,NaN
2,"102,en,/osoindex/data/objects/2025/2025-085t_2...",/osoindex/data/objects/2025/2025-085t_24498.html,2025-085T,false,,STARLINK 33886,,USA,false,2025-04-28,...,false,false,------,Not registered with the United Nations. Date o...,,NaN,NaN,NaN,NaN,NaN
3,"102,en,/osoindex/data/objects/2025/2025-085u_2...",/osoindex/data/objects/2025/2025-085u_24499.html,2025-085U,false,,STARLINK 33840,,USA,false,2025-04-28,...,false,false,------,Not registered with the United Nations. Date o...,,NaN,NaN,NaN,NaN,NaN
4,"102,en,/osoindex/data/objects/2025/2025-085v_2...",/osoindex/data/objects/2025/2025-085v_24500.html,2025-085V,false,,STARLINK 33851,,USA,false,2025-04-28,...,false,false,------,Not registered with the United Nations. Date o...,,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Inputs: df_unoosa and path. Process: save CSV. Output: data written to exports directory.
from pathlib import Path

# set your target directory and filename
output_dir = Path("/Users/aaeush/Desktop/Drive/Drive/Academics/Py Project/MyCode/OrbitIQ/exports")
output_dir.mkdir(parents=True, exist_ok=True)

out_path = output_dir / "unoosa_index_of_objects_launched_into_space.csv"
df_unoosa.to_csv(out_path, index=False, encoding="utf-8")
print(f"Saved CSV to: {out_path}")

Saved CSV to: /Users/aaeush/Desktop/Drive/Drive/Academics/Py Project/MyCode/OrbitIQ/exports/unoosa_index_of_objects_launched_into_space.csv


In [ ]:
# Inputs: column map. Process: define renames. Output: clearer df columns.
unoosa_rename_map = {
    "id": "id",
    "uri": "uri",

    "values.object.internationalDesignator_s1": "international_designator", #ID of object
    "values.object.internationalDesignator@official_s1": "international_designator_off", #True or False
    "values.object.nationalDesignator_s1": "national_designator",

    "values.object.nameOfSpaceObjectIno_s1": "space_object_name",

    "values.object.nameOfSpaceObjectO_s1": "space_object_name_2",

    "values.object.launch.stateOfRegistry_s1": "state_of_registry",
    "values.object.launch.stateOfRegistry@official_s1": "state_of_registry_off",

    "values.object.launch.dateOfLaunch_s1": "date_of_launch",
    "values.object.status.gsoLocation_s1": "gso_location",
    "values.object.unRegistration.unRegistered_s1": "un_registered",
    "values.en#object.status.objectStatus_s1": "status",
    "values.object.status@official_s1": "status_off",
    "values.object.status.dateOfDecay_s1": "date_of_decay",

    "values.object.launch.dateOfLaunch@official_s1":"date_of_launch_off" ,
    "values.object.status.dateOfDecay@official_s1":"date_of_decay_off" ,

    "values.object.functionOfSpaceObject_s1": "function",
    "values.object.remark_s1": "remarks",

    "values.object.status.webSite_s1": "external_website",

    "values.object.unRegistration.registrationDocuments.document@uri_s": "registration_doc",
    
    "values.object.unRegistration.registrationDocuments.document..document.symbol_s": "values.object.unRegistration.registrationDocuments.document..document.symbol_s",
    "values.object.status.gsoLocation@official_s1": "gso_location_off",
    "values.object.unRegistration.decayDocuments.document@uri_s": "decay_document_uri",
    "values.object.unRegistration.decayDocuments.document..document.symbol_s": "symbol",
}

In [ ]:
# Inputs: delete-list. Process: choose columns to drop. Output: leaner dataset.
unoosa_delete_list = [
    "values.object.unRegistration.registrationDocuments.document..document.symbol_s",
    "decay_document_uri",
]

In [ ]:
# Inputs: df_unoosa + drop-list. Process: drop cols. Output: cleaned columns and list removed.

cols_to_drop = [c for c in unoosa_delete_list if c in df_unoosa.columns]
df_unoosa.drop(columns=cols_to_drop, inplace=True)
print("Removed columns:", cols_to_drop)
df_unoosa.columns

### EDA Insights Summary

#### 1. Launch Date Patterns
- **Oldest launch:** 1957.
- **Most recent launch:** April 28 2025.
- **Peak launch activity:** 2021-01-24 with 131 recorded launches.

#### 2. Data Completeness
- Some `date_of_launch` fields are empty.
- GSO location missing for ~15,133 records — likely due to non-GEO orbits.
- Most objects lack a `date_of_decay`.
- Very few records have decay documentation.

#### 3. Identifiers & Registrations
- 100% of records have an international designator.
- 7,576 have a national designator; remainder missing.
- Most designators and registry states use official standardized forms.

#### 4. Duplication & Repetition
- International designator `2018-092` appears 106 times — likely a large multi-satellite deployment.
- Duplicate object names exist in dataset.

#### 5. Metadata Availability
- Very few objects have a website name field filled.
- Some objects have linked documentation.
- Common “Generic functions” classification indicates limited detail for many records.

#### 6. Special Columns of Interest
- `status_off` column should be investigated to determine the reasons for various statuses.

---
***Next Steps***
- Verify cause of missing launch dates.
- Confirm if repeated designators indicate multiple payloads per launch.
- Assess if “Generic functions” can be refined into specific mission categories.
- Explore each column and its values


In [ ]:
# Inputs: df_unoosa. Process: inspect categorical uniques. Output: quick value audit.
import pandas as pd
from pandas.api.types import is_object_dtype, is_categorical_dtype, is_string_dtype

# Identify categorical-like columns (object, string, or pandas categorical)
cat_cols = [
    col for col in df_unoosa.columns
    if is_categorical_dtype(df_unoosa[col])
    or is_object_dtype(df_unoosa[col])
    or is_string_dtype(df_unoosa[col])
]

# Map each categorical column to its list of unique non-null values
cat_uniques = {col: df_unoosa[col].dropna().unique().tolist() for col in cat_cols}

# Print
for col, vals in cat_uniques.items():
    print(f"{col} ({len(vals)} unique):")
    print(vals)
    print("-" * 60)

# Optional: treat low-cardinality numeric columns as categorical too (uncomment and adjust threshold)
# low_card_cols = [c for c in df_unoosa.columns
#                  if df_unoosa[c].nunique(dropna=True) <= 50]  # threshold
# for col in sorted(set(low_card_cols) - set(cat_cols)):
#     vals = df_unoosa[col].dropna().unique().tolist()
#     print(f"{col} ({len(vals)} unique):")
#     print(vals)
#     print("-" * 60)

In [ ]:
# Inputs: df_unoosa + export path. Process: save interim CSV. Output: raw export for backup.
df_unoosa.to_csv("/Users/aaeush/Desktop/Drive/Drive/Academics/Py Project/MyCode/OrbitIQ/exports/df_unoosa.csv", index=False)

In [ ]:
# Inputs: df_unoosa + maps. Process: drop/rename, parse dates. Output: cleaned df_unoosa.
# Ensure drops happen even if order changed
cols_to_drop2 = [c for c in unoosa_delete_list if c in df_unoosa.columns]
if cols_to_drop2:
    df_unoosa.drop(columns=cols_to_drop2, inplace=True)

# Apply friendly column names
df_unoosa.rename(columns=unoosa_rename_map, inplace=True)

# Parse dates if present
for col in ["date_of_launch", "date_of_decay"]:
    if col in df_unoosa.columns:
        df_unoosa[col] = pd.to_datetime(df_unoosa[col], errors="coerce")

print("Columns after clean:")
print(sorted(df_unoosa.columns))
print("Rows:", len(df_unoosa))


In [ ]:
# Inputs: df_unoosa. Process: save to processed folder. Output: processed CSV path.
from pathlib import Path

processed_dir = Path("/Users/aaeush/Desktop/Drive/Drive/Academics/Py Project/MyCode/OrbitIQ/orbitiq/data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

processed_path = processed_dir / "dn_unoosa.csv"
df_unoosa.to_csv(processed_path, index=False)
print(f"Saved processed CSV to: {processed_path}")
